# Unidade 4 - Bloco prático da Aula 01: CASH artesanal com Optuna

Monta à mão um espaço CASH: suggest_categorical escolhe o algoritmo e cada ramo abre os hiperparâmetros daquele algoritmo, buscado com TPE. Acompanhe como o orçamento de tentativas se concentra no ramo mais promissor ao longo da busca.

In [3]:
!pip install optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 425.6/425.6 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 265.9/265.9 kB 9.8 MB/s eta 0:00:00


In [4]:
import time
from collections import Counter

import numpy as np
import optuna

from sklearn.datasets import load_breast_cancer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import (
    RandomForestClassifier,
    GradientBoostingClassifier
)
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import (
    cross_val_score,
    StratifiedKFold
)


# ============================================================
# 1. CONFIGURAÇÕES GERAIS
# ============================================================

optuna.logging.set_verbosity(
    optuna.logging.WARNING
)

X, y = load_breast_cancer(
    return_X_y=True
)

cv = StratifiedKFold(
    n_splits=3,
    shuffle=True,
    random_state=42
)

N_TRIALS = 40


print("=" * 70)
print("OTIMIZAÇÃO DE ALGORITMO + HIPERPARÂMETROS")
print("=" * 70)

print(f"Amostras = {X.shape[0]}")
print(f"Características = {X.shape[1]}")
print(f"Classes = {np.bincount(y)}")
print(f"Validação cruzada = 3 folds")
print(f"Trials = {N_TRIALS}")


# ============================================================
# 2. FUNÇÃO OBJETIVO
# ============================================================

def objetivo(trial):

    # --------------------------------------------------------
    # Primeiro o Optuna escolhe QUAL algoritmo será usado
    # --------------------------------------------------------

    algoritmo = trial.suggest_categorical(
        "algoritmo",
        [
            "logistica",
            "floresta",
            "boosting"
        ]
    )


    # ========================================================
    # REGRESSÃO LOGÍSTICA
    # ========================================================

    if algoritmo == "logistica":

        C = trial.suggest_float(
            "logreg_C",
            1e-3,
            100,
            log=True
        )

        modelo = make_pipeline(

            # A regressão logística é sensível à escala
            StandardScaler(),

            LogisticRegression(
                C=C,
                max_iter=1000,
                random_state=42
            )
        )


    # ========================================================
    # RANDOM FOREST
    # ========================================================

    elif algoritmo == "floresta":

        n_estimators = trial.suggest_int(
            "rf_n_estimators",
            50,
            200
        )

        max_depth = trial.suggest_int(
            "rf_max_depth",
            3,
            12
        )

        min_samples_leaf = trial.suggest_int(
            "rf_min_leaf",
            1,
            8
        )

        modelo = RandomForestClassifier(
            n_estimators=n_estimators,
            max_depth=max_depth,
            min_samples_leaf=min_samples_leaf,

            random_state=42,

            # A paralelização ficará no cross_val_score
            n_jobs=1
        )


    # ========================================================
    # GRADIENT BOOSTING
    # ========================================================

    else:

        n_estimators = trial.suggest_int(
            "gb_n_estimators",
            50,
            200
        )

        learning_rate = trial.suggest_float(
            "gb_lr",
            0.01,
            0.3,
            log=True
        )

        max_depth = trial.suggest_int(
            "gb_max_depth",
            2,
            4
        )

        modelo = GradientBoostingClassifier(
            n_estimators=n_estimators,
            learning_rate=learning_rate,
            max_depth=max_depth,

            # Early stopping
            n_iter_no_change=10,
            validation_fraction=0.15,

            random_state=42
        )


    # ========================================================
    # VALIDAÇÃO CRUZADA
    # ========================================================

    scores = cross_val_score(
        modelo,
        X,
        y,

        cv=cv,

        scoring="f1",

        # Os 3 folds podem rodar em paralelo
        n_jobs=-1
    )


    # Guarda informações extras dentro do trial
    trial.set_user_attr(
        "f1_fold_1",
        scores[0]
    )

    trial.set_user_attr(
        "f1_fold_2",
        scores[1]
    )

    trial.set_user_attr(
        "f1_fold_3",
        scores[2]
    )

    trial.set_user_attr(
        "desvio_padrao",
        scores.std()
    )


    return scores.mean()


# ============================================================
# 3. MOSTRAR CADA TRIAL
# ============================================================

def mostrar_trial(study, trial):

    algoritmo = trial.params[
        "algoritmo"
    ]

    simbolo = ""

    if trial.number == study.best_trial.number:
        simbolo = "  <-- MELHOR ATÉ AGORA"

    print(
        f"\nTrial {trial.number + 1:02d}"
        f" | {algoritmo:10s}"
        f" | F1 = {trial.value:.4f}"
        f" | Melhor = {study.best_value:.4f}"
        f"{simbolo}"
    )

    print(
        f"    parâmetros = "
        f"{trial.params}"
    )

    print(
        f"    folds = "
        f"{trial.user_attrs['f1_fold_1']:.4f}, "
        f"{trial.user_attrs['f1_fold_2']:.4f}, "
        f"{trial.user_attrs['f1_fold_3']:.4f}"
    )


# ============================================================
# 4. CRIANDO O ESTUDO
# ============================================================

sampler = optuna.samplers.TPESampler(
    seed=42
)

estudo = optuna.create_study(
    direction="maximize",
    sampler=sampler
)


# ============================================================
# 5. EXECUTANDO A OTIMIZAÇÃO
# ============================================================

print("\n")
print("=" * 70)
print("INICIANDO BUSCA")
print("=" * 70)

inicio = time.time()

estudo.optimize(
    objetivo,
    n_trials=N_TRIALS,
    callbacks=[
        mostrar_trial
    ]
)

tempo_total = (
    time.time() - inicio
)


# ============================================================
# 6. RESULTADO GERAL
# ============================================================

print("\n")
print("=" * 70)
print("RESULTADO GERAL")
print("=" * 70)

print(
    f"Melhor F1 = "
    f"{estudo.best_value:.4f}"
)

print(
    f"Melhor algoritmo = "
    f"{estudo.best_params['algoritmo']}"
)

print(
    f"\nMelhor configuração"
)

for parametro, valor in estudo.best_params.items():

    print(
        f"    {parametro:20s} = {valor}"
    )

print(
    f"\nTempo total = "
    f"{tempo_total:.1f} segundos"
)


# ============================================================
# 7. QUANTAS VEZES CADA ALGORITMO FOI TESTADO
# ============================================================

print("\n")
print("=" * 70)
print("TENTATIVAS POR ALGORITMO")
print("=" * 70)

contagem = Counter(
    trial.params["algoritmo"]
    for trial in estudo.trials
)

for algoritmo, quantidade in contagem.items():

    percentual = (
        quantidade
        / len(estudo.trials)
        * 100
    )

    print(
        f"{algoritmo:10s} "
        f"= {quantidade:2d} trials "
        f"({percentual:.1f}%)"
    )


# ============================================================
# 8. MELHOR RESULTADO DE CADA ALGORITMO
# ============================================================

print("\n")
print("=" * 70)
print("MELHOR RESULTADO POR ALGORITMO")
print("=" * 70)

algoritmos = [
    "logistica",
    "floresta",
    "boosting"
]

for algoritmo in algoritmos:

    trials_algoritmo = [
        trial
        for trial in estudo.trials
        if (
            trial.value is not None
            and trial.params["algoritmo"]
            == algoritmo
        )
    ]

    if len(trials_algoritmo) == 0:
        continue

    melhor = max(
        trials_algoritmo,
        key=lambda trial: trial.value
    )

    print(
        f"\n{algoritmo.upper()}"
    )

    print(
        f"Melhor F1 = "
        f"{melhor.value:.4f}"
    )

    print(
        f"Trial = "
        f"{melhor.number + 1}"
    )

    print(
        "Parâmetros"
    )

    for parametro, valor in melhor.params.items():

        if parametro != "algoritmo":

            print(
                f"    {parametro} = "
                f"{valor}"
            )


# ============================================================
# 9. MÉDIA DE DESEMPENHO POR ALGORITMO
# ============================================================

print("\n")
print("=" * 70)
print("DESEMPENHO MÉDIO")
print("=" * 70)

for algoritmo in algoritmos:

    valores = [
        trial.value
        for trial in estudo.trials
        if (
            trial.value is not None
            and trial.params["algoritmo"]
            == algoritmo
        )
    ]

    if len(valores) == 0:
        continue

    print(
        f"{algoritmo:10s}"
        f" | média = {np.mean(valores):.4f}"
        f" | melhor = {np.max(valores):.4f}"
        f" | pior = {np.min(valores):.4f}"
    )


# ============================================================
# 10. TOP 5 TRIALS
# ============================================================

print("\n")
print("=" * 70)
print("TOP 5 CONFIGURAÇÕES")
print("=" * 70)

trials_ordenados = sorted(
    [
        trial
        for trial in estudo.trials
        if trial.value is not None
    ],

    key=lambda trial: trial.value,

    reverse=True
)

for posicao, trial in enumerate(
    trials_ordenados[:5],
    start=1
):

    print(
        f"\n#{posicao}"
        f" | Trial {trial.number + 1}"
        f" | {trial.params['algoritmo']}"
        f" | F1 = {trial.value:.4f}"
    )

    print(
        f"    {trial.params}"
    )

OTIMIZAÇÃO DE ALGORITMO + HIPERPARÂMETROS
Amostras = 569
Características = 30
Classes = [212 357]
Validação cruzada = 3 folds
Trials = 40


INICIANDO BUSCA

Trial 01 | floresta   | F1 = 0.9617 | Melhor = 0.9617  <-- MELHOR ATÉ AGORA
    parâmetros = {'algoritmo': 'floresta', 'rf_n_estimators': 140, 'rf_max_depth': 4, 'rf_min_leaf': 2}
    folds = 0.9705, 0.9398, 0.9748

Trial 02 | floresta   | F1 = 0.9522 | Melhor = 0.9617
    parâmetros = {'algoritmo': 'floresta', 'rf_n_estimators': 156, 'rf_max_depth': 3, 'rf_min_leaf': 8}
    folds = 0.9576, 0.9286, 0.9705

Trial 03 | logistica  | F1 = 0.9611 | Melhor = 0.9617
    parâmetros = {'algoritmo': 'logistica', 'logreg_C': 0.008260808399079604}
    folds = 0.9754, 0.9407, 0.9672

Trial 04 | floresta   | F1 = 0.9602 | Melhor = 0.9617
    parâmetros = {'algoritmo': 'floresta', 'rf_n_estimators': 93, 'rf_max_depth': 9, 'rf_min_leaf': 2}
    folds = 0.9705, 0.9398, 0.9705

Trial 05 | boosting   | F1 = 0.9544 | Melhor = 0.9617
    parâmetros = {